# Phụ lục A.1 — Kiểm chứng dữ liệu, làm sạch, chia tập và EDA

**Phụ trách:** Nguyễn Văn Hoan  
**Đề tài:** Phân tích dữ liệu và xếp hạng nguy cơ gian lận trong giao dịch thẻ

## Mục tiêu

Notebook này thực hiện phần chuẩn bị dữ liệu của ECCC-LITE:

1. kiểm chứng `creditcard.csv` bằng kích thước, SHA-256 và schema;
2. kiểm tra missing, infinity, phân bố `Class` và exact duplicates;
3. tạo `source_row`, loại bản ghi trùng trước khi chia;
4. tạo `LogAmount = log1p(Amount)`;
5. chia train/validation/test 60/20/20 có stratify và seed 42;
6. EDA chỉ trên train;
7. xuất split, audit table, feature contract và bốn hình dùng trong báo cáo.

**Giới hạn:** notebook không huấn luyện mô hình, không chọn threshold và không dùng
test để chọn feature. Dữ liệu tổng hợp chỉ được dùng trong smoke test, không dùng
để ghi kết quả báo cáo.


## 1. Thiết lập môi trường và đường dẫn

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root() -> Path:
    configured = os.environ.get("A1_PROJECT_ROOT")
    if configured:
        return Path(configured).expanduser().resolve()
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "a1_utils.py").is_file():
            return candidate
    raise FileNotFoundError("Không tìm thấy thư mục gốc chứa src/a1_utils.py")


PROJECT_ROOT = find_project_root()
OUTPUT_ROOT = Path(os.environ.get("A1_OUTPUT_ROOT", PROJECT_ROOT)).expanduser().resolve()
RAW_PATH = Path(
    os.environ.get("A1_DATA_PATH", PROJECT_ROOT / "data" / "raw" / "creditcard.csv")
).expanduser().resolve()
STRICT_DATASET = os.environ.get("A1_STRICT_DATASET", "1").strip().lower() not in {"0", "false", "no"}

PROCESSED_DIR = OUTPUT_ROOT / "data" / "processed"
TABLE_DIR = OUTPUT_ROOT / "outputs" / "tables"
FIGURE_DIR = OUTPUT_ROOT / "outputs" / "figures"
for directory in (PROCESSED_DIR, TABLE_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.a1_utils import (
    EXPECTED_CLEAN_FRAUD,
    EXPECTED_CLEAN_ROWS,
    EXPECTED_DUPLICATES,
    ID_COLUMN,
    MODEL_FEATURE_COLUMNS,
    RANDOM_STATE,
    TARGET,
    assert_split_contract,
    atomic_write_csv,
    atomic_write_json,
    build_audit_table,
    clean_and_add_features,
    model_features,
    sha256_file,
    split_summary,
    stratified_split,
    validate_raw_dataframe,
    verify_raw_file,
)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

print(f"PROJECT_ROOT   : {PROJECT_ROOT}")
print(f"RAW_PATH       : {RAW_PATH}")
print(f"OUTPUT_ROOT    : {OUTPUT_ROOT}")
print(f"STRICT_DATASET : {STRICT_DATASET}")

## 2. Kiểm chứng nguồn và đọc dữ liệu

Ở chế độ chuẩn, notebook dừng ngay nếu kích thước hoặc SHA-256 không khớp. Nếu
thiếu dữ liệu, chạy `python scripts/get_data.py` tại thư mục gốc dự án.


In [ ]:
file_info = verify_raw_file(RAW_PATH, strict=STRICT_DATASET)
raw = pd.read_csv(RAW_PATH)
raw_summary = validate_raw_dataframe(raw, strict=STRICT_DATASET)

source_check = pd.DataFrame(
    {
        "thuoc_tinh": ["Đường dẫn", "Kích thước (byte)", "SHA-256", "Shape", "Class 1", "Exact duplicates"],
        "gia_tri": [
            file_info["path"],
            file_info["file_size_bytes"],
            file_info["sha256"],
            str(raw.shape),
            raw_summary["fraud"],
            raw_summary["exact_duplicates"],
        ],
    }
)
display(source_check)
display(raw.head())

## 3. Kiểm tra schema và chất lượng dữ liệu

In [ ]:
quality_table = pd.DataFrame(
    {
        "chi_tieu": ["Số dòng", "Số cột", "Missing", "Infinity", "Class 0", "Class 1", "Fraud rate"],
        "gia_tri": [
            len(raw),
            raw.shape[1],
            int(raw.isna().sum().sum()),
            int(np.isinf(raw.to_numpy(dtype=np.float64, copy=False)).sum()),
            int((raw[TARGET] == 0).sum()),
            int((raw[TARGET] == 1).sum()),
            float(raw[TARGET].mean()),
        ],
    }
)
display(quality_table)

dtype_table = raw.dtypes.rename("dtype").astype(str).to_frame()
display(dtype_table.T)

## 4. Tạo `source_row`, bỏ exact duplicates và tạo `LogAmount`

`source_row` được tạo trước mọi thao tác lọc. Duplicate được xác định trên đúng
31 cột gốc; giữ lần xuất hiện đầu tiên. `LogAmount` là phép biến đổi theo từng
dòng nên không học tham số từ validation/test.


In [ ]:
clean, duplicate_count = clean_and_add_features(raw)

if STRICT_DATASET:
    assert duplicate_count == EXPECTED_DUPLICATES
    assert len(clean) == EXPECTED_CLEAN_ROWS
    assert int(clean[TARGET].sum()) == EXPECTED_CLEAN_FRAUD

cleaning_summary = pd.DataFrame(
    {
        "giai_doan": ["Raw", "Sau bỏ trùng"],
        "so_dong": [len(raw), len(clean)],
        "class_0": [int((raw[TARGET] == 0).sum()), int((clean[TARGET] == 0).sum())],
        "class_1": [int(raw[TARGET].sum()), int(clean[TARGET].sum())],
    }
)
display(cleaning_summary)
display(clean[[ID_COLUMN, "Amount", "LogAmount", TARGET]].head())

## 5. Chia train/validation/test

- Bước 1: tách test 20% từ dữ liệu sạch.
- Bước 2: tách validation bằng 25% của phần 80% còn lại.
- Cả hai bước đều dùng `stratify=Class`, `random_state=42`.
- EDA và mọi quyết định tiếp theo chỉ được dùng train/validation; test được giữ
  kín cho đánh giá cuối của nhóm.


In [ ]:
splits = stratified_split(clean, random_state=RANDOM_STATE)
train = splits["train"]
validation = splits["validation"]
test = splits["test"]

summary = split_summary(splits)
display(summary.style.format({"fraud_rate": "{:.6%}"}))

if STRICT_DATASET:
    expected_counts = {
        "train": (170_235, 284),
        "validation": (56_745, 94),
        "test": (56_746, 95),
    }
    for name, (expected_rows, expected_fraud) in expected_counts.items():
        frame = splits[name]
        assert (len(frame), int(frame[TARGET].sum())) == (expected_rows, expected_fraud)

assert_split_contract(splits, expected_total=len(clean))

## 6. Xuất split, audit table và feature contract

In [ ]:
for name, frame in splits.items():
    atomic_write_csv(frame, PROCESSED_DIR / f"{name}.csv")

audit = build_audit_table(
    file_info=file_info,
    raw_summary=raw_summary,
    clean=clean,
    duplicate_count=duplicate_count,
    splits=splits,
)
atomic_write_csv(audit, TABLE_DIR / "data_audit.csv")
atomic_write_csv(summary, TABLE_DIR / "split_summary.csv")

train_class_summary = (
    train.groupby(TARGET, as_index=False)
    .agg(
        rows=(TARGET, "size"),
        amount_median=("Amount", "median"),
        amount_mean=("Amount", "mean"),
        log_amount_median=("LogAmount", "median"),
        time_min=("Time", "min"),
        time_max=("Time", "max"),
    )
)
atomic_write_csv(train_class_summary, TABLE_DIR / "train_class_summary.csv")

feature_frame = model_features(train)
feature_contract = {
    "target": TARGET,
    "identifier_not_a_feature": ID_COLUMN,
    "features": list(feature_frame.columns),
    "fit_scope": "train only",
    "validation_role": "model and threshold selection",
    "test_role": "single final evaluation",
}
atomic_write_json(feature_contract, TABLE_DIR / "feature_contract.json")
atomic_write_json(
    {
        **file_info,
        "raw_rows": len(raw),
        "clean_rows": len(clean),
        "exact_duplicates_removed": duplicate_count,
        "random_state": RANDOM_STATE,
    },
    TABLE_DIR / "data_source_manifest.json",
)

display(audit)
display(train_class_summary)

## 7. EDA chỉ trên train

Bốn hình trả lời bốn câu hỏi đã nêu trong Chương 2: mức mất cân bằng; phân bố
Amount; phân bố Time tương đối; và các biến có liên hệ thống kê với Class. Các
hình không được dùng để suy ra quan hệ nhân quả hoặc gán ý nghĩa nghiệp vụ cho
`V1`–`V28`.


In [ ]:
def save_current_figure(filename: str) -> None:
    path = FIGURE_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"Đã lưu: {path}")


class_counts = train[TARGET].value_counts().sort_index()
colors = ["#4C78A8", "#E45756"]
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(["Class 0", "Class 1"], class_counts.values, color=colors)
ax.set_title("Phân bố lớp trên tập train")
ax.set_ylabel("Số giao dịch")
for bar, count in zip(bars, class_counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{count:,}\n({count / len(train):.4%})",
        ha="center",
        va="bottom",
    )
save_current_figure("class_distribution.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for class_value, color in zip((0, 1), colors):
    values = train.loc[train[TARGET] == class_value, "LogAmount"]
    axes[0].hist(values, bins=60, density=True, alpha=0.55, color=color, label=f"Class {class_value}")
axes[0].set_title("Phân bố LogAmount theo lớp")
axes[0].set_xlabel("LogAmount = log1p(Amount)")
axes[0].set_ylabel("Mật độ")
axes[0].legend()

axes[1].boxplot(
    [train.loc[train[TARGET] == 0, "LogAmount"], train.loc[train[TARGET] == 1, "LogAmount"]],
    showfliers=False,
)
axes[1].set_xticks([1, 2])
axes[1].set_xticklabels(["Class 0", "Class 1"])
axes[1].set_title("LogAmount theo lớp (không vẽ điểm ngoại lai)")
axes[1].set_ylabel("LogAmount")
save_current_figure("amount_by_class.png")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for class_value, color in zip((0, 1), colors):
    hours = train.loc[train[TARGET] == class_value, "Time"] / 3600
    ax.hist(hours, bins=48, density=True, alpha=0.55, color=color, label=f"Class {class_value}")
ax.set_title("Phân bố Time tương đối theo lớp trên train")
ax.set_xlabel("Giờ tương đối từ giao dịch đầu tiên")
ax.set_ylabel("Mật độ")
ax.legend()
save_current_figure("time_by_class.png")

In [ ]:
correlations = (
    train[MODEL_FEATURE_COLUMNS + [TARGET]]
    .corr(numeric_only=True)[TARGET]
    .drop(TARGET)
)
selected = correlations.loc[correlations.abs().nlargest(12).index].sort_values()
selected_table = (
    selected.rename("correlation_with_class")
    .rename_axis("feature")
    .reset_index()
)
atomic_write_csv(selected_table, TABLE_DIR / "selected_correlations.csv")

fig, ax = plt.subplots(figsize=(8, 5.5))
bar_colors = ["#4C78A8" if value < 0 else "#E45756" for value in selected.values]
ax.barh(selected.index, selected.values, color=bar_colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("12 biến có |tương quan| lớn nhất với Class trên train")
ax.set_xlabel("Hệ số tương quan Pearson")
save_current_figure("selected_correlations.png")
display(selected_table)

## 8. Kiểm tra cuối và bàn giao

In [ ]:
assert_split_contract(splits, expected_total=len(clean))
assert TARGET not in feature_frame.columns
assert ID_COLUMN not in feature_frame.columns
assert file_info["sha256"] == sha256_file(RAW_PATH), "Raw đã thay đổi trong lúc chạy"

required_files = [
    *(PROCESSED_DIR / f"{name}.csv" for name in ("train", "validation", "test")),
    TABLE_DIR / "data_audit.csv",
    TABLE_DIR / "split_summary.csv",
    TABLE_DIR / "train_class_summary.csv",
    TABLE_DIR / "feature_contract.json",
    TABLE_DIR / "data_source_manifest.json",
    TABLE_DIR / "selected_correlations.csv",
    *(FIGURE_DIR / name for name in (
        "class_distribution.png",
        "amount_by_class.png",
        "time_by_class.png",
        "selected_correlations.png",
    )),
]
missing_outputs = [str(path) for path in required_files if not path.is_file()]
assert not missing_outputs, f"Thiếu đầu ra: {missing_outputs}"

print("[OK] Raw không bị sửa; SHA-256 giữ nguyên.")
print("[OK] Ba split đúng schema, không giao nhau và bảo toàn số dòng.")
print("[OK] EDA chỉ dùng train; Class/source_row không nằm trong feature contract.")
print("[OK] A.1 sẵn sàng chạy scripts/verify_a1_outputs.py và bàn giao.")

## Kết luận ngắn cho A.1

Notebook tạo một nguồn dữ liệu đã kiểm chứng và ba split cố định để các thành
viên dùng chung. Mất cân bằng lớp phải được báo cáo bằng prevalence và được xử
lý ở bước mô hình/đánh giá bằng metric phù hợp; bản thân EDA không chứng minh
nguyên nhân gian lận. Test không được mở để sửa feature hoặc chọn mô hình.
